# OpenJury Quickstart

Explore `AgentEvalResult` output. Early cells run **offline** (no API keys). The last cell optionally runs live jurors.

In [ ]:
# %pip install openjury  # uncomment if not installed
import sys
from pathlib import Path

repo_src = Path("..").resolve() / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

from openjury import ResultFormatter
from openjury.output_format import AgentEvalResult, CriterionEvaluation, TrialResult
from openjury.scoring import JurorScore, ScoredMetrics

In [ ]:
# Offline demo — same fixture data as examples/hello_world/
PROMPT = "Write a helpful response to demonstrate the system."
AGENT_TEXT = (
    "OpenJury evaluates agent responses using a panel of LLM judges."
)

juror_scores = [
    JurorScore("Expert Juror", 2.0, {"factuality": 4.5, "clarity": 4.0},
               {"factuality": "Accurate.", "clarity": "Clear."}),
    JurorScore("General Juror", 1.0, {"factuality": 4.0, "clarity": 4.5},
               {"factuality": "Mostly accurate.", "clarity": "Easy to follow."}),
]
metrics = ScoredMetrics(
    weighted_mean=4.17, mean=4.25, median=4.25,
    min_score=4.0, max_score=4.5, harmonic_mean=4.21,
    weakest_link=0.8, juror_agreement=0.92,
)
criteria_evaluations = {
    "factuality": CriterionEvaluation(
        weighted_mean_score=4.33, min_juror_score=4.0, max_juror_score=4.5,
        juror_agreement=0.94, weight=2.0,
    ),
    "clarity": CriterionEvaluation(
        weighted_mean_score=4.17, min_juror_score=4.0, max_juror_score=4.5,
        juror_agreement=0.91, weight=1.0,
    ),
}
result = AgentEvalResult(
    jury_name="Notebook Demo",
    prompt=PROMPT,
    score_scale=5,
    composite_score=metrics.weighted_mean,
    normalized_composite_score=metrics.weighted_mean / 5,
    scored_metrics=metrics,
    criteria_evaluations=criteria_evaluations,
    juror_scores=juror_scores,
    trial_results=[TrialResult(1, AGENT_TEXT, metrics, criteria_evaluations, juror_scores)],
)
print(ResultFormatter.format_result(result))

In [ ]:
# Inspect key fields
print("composite_score:", result.composite_score)
print("normalized:", result.normalized_composite_score)
print("juror_agreement:", result.scored_metrics.juror_agreement)
for name, crit in result.criteria_evaluations.items():
    print(f"  {name}: {crit.weighted_mean_score:.2f} (agreement={crit.juror_agreement:.2f})")

In [ ]:
# Optional: live evaluation (requires OPENAI_API_KEY)
import os
from openjury import AgentResponse, JuryConfig, OpenJury

if os.environ.get("OPENAI_API_KEY"):
    cfg_path = Path("../examples/hello_world/config.json").resolve()
    jury = OpenJury(JuryConfig.from_json_file(str(cfg_path)))
    live = jury.score_existing_response(
        prompt=PROMPT,
        agent_response=AgentResponse(content=AGENT_TEXT),
    )
    print(f"Live composite_score: {live.composite_score:.2f}")
else:
    print("Set OPENAI_API_KEY to run live jurors.")